# CNEE QLoRA Training Notebook for Google Colab (T4)

This notebook installs dependencies, mounts Google Drive, clones a chosen Qwen VLM base model from Hugging Face with Git LFS, prepares the CNEE dataset, fine-tunes a Vision-Language model with Unsloth and QLoRA, evaluates the fine-tuned model, and saves a merged output for inference.

Recommended for T4: `Qwen/Qwen3-VL-2B-Instruct` or `Qwen/Qwen3.5-2B-Instruct`.


## 1. Mount Google Drive

Mount your Google Drive and set the working root to `/content/drive/MyDrive/CNEE`. The notebook assumes the dataset and saved model files will reside under that folder.


In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive mounted successfully.')
except Exception as e:
    print('Not running in Colab or mount failed:', str(e))

DRIVE_ROOT = '/content/drive/MyDrive/CNEE'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


## 2. Install dependencies

Install system tools, Git LFS, PyTorch, Unsloth, and training dependencies for a Colab T4 environment.


In [ ]:
!apt-get update -y && apt-get install -y git-lfs
!git lfs install
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!python -m pip install --quiet transformers accelerate peft bitsandbytes trl datasets pillow matplotlib qwen-vl-utils unsloth-zoo
!python -m pip install --quiet "unsloth @ git+https://github.com/unslothai/unsloth.git"


## 3. Configure model, dataset, and output paths

Set the model choice, local Drive paths, and output directories. You can change `MODEL_CHOICE` to another supported base model.


In [ ]:
import os

DRIVE_ROOT = '/content/drive/MyDrive/CNEE'
MODEL_CHOICE = 'Qwen/Qwen3-VL-2B-Instruct'  # Change to 'Qwen/Qwen3.5-2B-Instruct' if desired
MODEL_DIR = os.path.join(DRIVE_ROOT, 'models', MODEL_CHOICE.replace('/', '_'))
DATASET_JSON = os.path.join(DRIVE_ROOT, 'dataset', 'dataset_FINAL_100casos.json')
OUTPUT_DIR = os.path.join(DRIVE_ROOT, 'output', 'colab', MODEL_CHOICE.replace('/', '_'))
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('MODEL_CHOICE =', MODEL_CHOICE)
print('MODEL_DIR =', MODEL_DIR)
print('DATASET_JSON =', DATASET_JSON)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 4. Clone the base model with Git LFS

Clone the chosen base model from Hugging Face into your mounted Drive, using Git LFS to download large files.


In [ ]:
import os
import subprocess

def run_shell(command):
    print('>>>', command)
    completed = subprocess.run(command, shell=True, text=True, capture_output=True)
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError(f'Command failed with code {completed.returncode}: {command}')

if not os.path.isdir(MODEL_DIR) or not os.listdir(MODEL_DIR):
    os.makedirs(os.path.dirname(MODEL_DIR), exist_ok=True)
    run_shell(f'git clone https://huggingface.co/{MODEL_CHOICE} "{MODEL_DIR}"')
    run_shell(f'git -C "{MODEL_DIR}" lfs pull')
else:
    print('Base model directory already exists. Skipping clone.')

if not os.path.isfile(DATASET_JSON):
    raise FileNotFoundError(
        f'Dataset file not found at {DATASET_JSON}. Upload dataset_FINAL_100casos.json to {os.path.dirname(DATASET_JSON)} in Google Drive.'
    )
print('Model and dataset checks passed.')


## 5. Verify GPU and environment

Confirm the Colab GPU and PyTorch setup before training.


In [ ]:
import torch
print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Total VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)
    print('CUDA device count:', torch.cuda.device_count())


## 6. Prepare the dataset

Load the JSON case dataset, build the chat-style examples, and create a stratified train/validation split.


In [ ]:
import json
import random
from collections import defaultdict
from datasets import Dataset

MAX_IMAGES = 3  # Keep small for T4 memory. Increase only if you confirm fit.

with open(DATASET_JSON, 'r', encoding='utf-8') as f:
    dataset_json = json.load(f)
casos = dataset_json['casos']

def extract_text(message_block):
    for item in message_block.get('content', []):
        if item.get('type') == 'text' and item.get('text'):
            return item['text']
    return ''

def build_example(caso):
    imagenes = caso['images'][:MAX_IMAGES]
    rutas = [os.path.join(DRIVE_ROOT, img) if not os.path.isabs(img) else img for img in imagenes]
    texto_user = extract_text(caso['messages'][0])
    texto_assistant = extract_text(caso['messages'][1])
    return {
        'messages': [
            {
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': ruta} for ruta in rutas
                ] + [{'type': 'text', 'text': texto_user}]
            },
            {
                'role': 'assistant',
                'content': [{'type': 'text', 'text': texto_assistant}]
            },
        ]
    }

random.seed(42)
groups = defaultdict(list)
for idx, caso in enumerate(casos):
    groups[caso['metadata']['label']].append(idx)

val_indices = []
for label, indices in groups.items():
    n_val = max(1, int(len(indices) * 0.1))
    val_indices.extend(random.sample(indices, n_val))

val_set = set(val_indices)
train_indices = [i for i in range(len(casos)) if i not in val_set]

train_examples = [build_example(casos[i]) for i in train_indices]
eval_examples = [build_example(casos[i]) for i in val_indices]

train_dataset = Dataset.from_list(train_examples)
eval_dataset = Dataset.from_list(eval_examples)

print(f'Train examples: {len(train_dataset)}')
print(f'Validation examples: {len(eval_dataset)}')
print('Train labels:', {k: sum(1 for i in train_indices if casos[i]['metadata']['label'] == k) for k in groups})
print('Val labels:  ', {k: sum(1 for i in val_indices if casos[i]['metadata']['label'] == k) for k in groups})


## 7. Fine-tune with Unsloth + QLoRA

Train the model using 4-bit quantization, frozen base weights, and LoRA adapters.


In [ ]:
import gc
import os
import torch
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

torch._dynamo.config.disable = True
os.environ['TORCHINDUCTOR_COMPILE_THREADS'] = '1'

NUM_EPOCHS = 3  # Start small for the T4; increase when stable.
BATCH_SIZE = 1
GRAD_ACCUM = 16

print('Loading base model...')
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_DIR,
    load_in_4bit=True,
    use_gradient_checkpointing='unsloth',
)
tokenizer.model_max_length = 8192

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    random_state=42,
)

print('Preparing trainer...')
FastVisionModel.for_training(model)
steps_per_epoch = max(1, len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        output_dir=os.path.join(OUTPUT_DIR, 'final'),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=True,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=1e-4,
        bf16=True,
        logging_steps=steps_per_epoch,
        save_strategy='epoch',
        save_total_limit=2,
        eval_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
        per_device_eval_batch_size=1,
        eval_accumulation_steps=1,
        remove_unused_columns=False,
        dataset_text_field='text',
        dataset_kwargs={'skip_prepare_dataset': True},
        max_seq_length=8192,
    ),
)

torch.cuda.empty_cache()
gc.collect()

print('Starting fine-tuning...')
trainer.train()

print('Saving final model and tokenizer...')
model.save_pretrained(os.path.join(OUTPUT_DIR, 'final'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'final'))
print('Fine-tuning complete. Saved to', os.path.join(OUTPUT_DIR, 'final'))


## 8. Evaluate the fine-tuned model

Run inference on the validation split and compute case-level metrics.


In [ ]:
import json
import os
import re
import torch
from PIL import Image
from unsloth import FastVisionModel

MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'final')
print('Loading trained model from', MODEL_OUTPUT_DIR)
model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_OUTPUT_DIR,
    load_in_4bit=True,
)
tokenizer.model_max_length = 8192
FastVisionModel.for_inference(model)

def extract_decision(text):
    match = re.search(r'\"decision\"\s*:\s*\"(APROBADO|RECHAZADO)\"', text)
    if match:
        return match.group(1)
    if 'APROBADO' in text.upper():
        return 'APROBADO'
    if 'RECHAZADO' in text.upper():
        return 'RECHAZADO'
    return 'DESCONOCIDO'

def infer_case(caso):
    imagenes = caso['images'][:MAX_IMAGES]
    rutas = [os.path.join(DRIVE_ROOT, img) if not os.path.isabs(img) else img for img in imagenes]
    texto_user = extract_text(caso['messages'][0])
    messages = [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': ruta} for ruta in rutas
            ] + [{'type': 'text', 'text': texto_user}]
        }
    ]
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    images = [Image.open(ruta).convert('RGB') for ruta in rutas]
    inputs = tokenizer(
        text=text_input,
        images=images,
        return_tensors='pt',
        truncation=True,
        max_length=4096,
    ).to('cuda')

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False,
        )

    generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

eval_casos = [casos[i] for i in val_indices]
results = []
total_exact = 0
total_vqa = 0
for i, caso in enumerate(eval_casos, 1):
    print(f'[{i}/{len(eval_casos)}] Caso {caso["id"]}')
    ground_truth = extract_text(caso['messages'][1])
    label_real = caso['metadata']['label']
    prediction = infer_case(caso)
    label_pred = extract_decision(prediction)
    exact = ground_truth.strip() == prediction.strip()
    vqa_ok = label_pred == label_real
    total_exact += int(exact)
    total_vqa += int(vqa_ok)
    results.append({
        'id': caso['id'],
        'label_real': label_real,
        'label_pred': label_pred,
        'exact_match': exact,
        'vqa_ok': vqa_ok,
        'prediction': prediction,
    })
    print(f'    Real={label_real} Pred={label_pred} Exact={exact} VQA={vqa_ok}')

n_val = len(eval_casos)
accuracy_vqa = total_vqa / n_val * 100
accuracy_exact = total_exact / n_val * 100
tp = sum(1 for r in results if r['label_real'] == 'APROBADO' and r['label_pred'] == 'APROBADO')
tn = sum(1 for r in results if r['label_real'] == 'RECHAZADO' and r['label_pred'] == 'RECHAZADO')
fp = sum(1 for r in results if r['label_real'] == 'RECHAZADO' and r['label_pred'] == 'APROBADO')
fn = sum(1 for r in results if r['label_real'] == 'APROBADO' and r['label_pred'] == 'RECHAZADO')
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print('--- Evaluation summary ---')
print(f'VQA Accuracy: {total_vqa}/{n_val} = {accuracy_vqa:.2f}%')
print(f'Exact Match: {total_exact}/{n_val} = {accuracy_exact:.2f}%')
print(f'Precision: {precision:.3f} Recall: {recall:.3f} F1: {f1:.3f}')
print('Confusion matrix TP/TN/FP/FN:', tp, tn, fp, fn)

metrics = {
    'n_val': n_val,
    'vqa_accuracy': accuracy_vqa,
    'exact_match': accuracy_exact,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'confusion_matrix': {'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn},
    'results': results,
}
with open(os.path.join(OUTPUT_DIR, 'metricas_validacion.json'), 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('Saved validation metrics to', os.path.join(OUTPUT_DIR, 'metricas_validacion.json'))


## 9. Merge fine-tuned weights and save merged output

Save a merged version of the fine-tuned model so it is easier to load for inference.


In [ ]:
import os
from unsloth import FastVisionModel

MERGED_DIR = os.path.join(OUTPUT_DIR, 'merged')
os.makedirs(MERGED_DIR, exist_ok=True)

try:
    from peft import PeftModel
    base_model, _ = FastVisionModel.from_pretrained(
        model_name=MODEL_DIR,
        load_in_4bit=True,
        use_gradient_checkpointing='unsloth',
    )
    merged_model = PeftModel.from_pretrained(base_model, os.path.join(OUTPUT_DIR, 'final'))
    merged_model.save_pretrained(MERGED_DIR)
    print('Merged model saved to', MERGED_DIR)
except Exception as exc:
    print('PEFT merge path failed:', exc)
    print('Saving the trained model weights directly to the merged directory instead.')
    model.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)


## 10. Optional: convert merged model to GGUF

The notebook saves a merged PyTorch/LoRA model. Converting to GGUF requires a separate conversion tool such as `llama.cpp`/`gguf` support for Qwen models.

If you have a conversion tool available, point it at `OUTPUT_DIR/merged` and save into `OUTPUT_DIR/merged_gguf`.


In [ ]:
import os
print('Merged model directory:', os.path.join(OUTPUT_DIR, 'merged'))
print('If you want to convert to GGUF, use a compatible conversion utility outside this notebook or add a conversion tool here.')
print('Example conversion command:')
print('  python -m llama_cpp.convert --model-type qwen --input', os.path.join(OUTPUT_DIR, 'merged'), '--output', os.path.join(OUTPUT_DIR, 'merged_gguf'))


## 11. Example inference with the fine-tuned model

Run a quick sample inference using the trained model and one case from validation.


In [ ]:
import random
from PIL import Image
from unsloth import FastVisionModel

sample_model_dir = os.path.join(OUTPUT_DIR, 'final')
model, tokenizer = FastVisionModel.from_pretrained(sample_model_dir, load_in_4bit=True)
tokenizer.model_max_length = 8192
FastVisionModel.for_inference(model)

sample_caso = casos[random.choice(val_indices)]
imagenes = sample_caso['images'][:MAX_IMAGES]
rutas = [os.path.join(DRIVE_ROOT, img) if not os.path.isabs(img) else img for img in imagenes]
texto_user = extract_text(sample_caso['messages'][0])
messages = [
    {
        'role': 'user',
        'content': [
            {'type': 'image', 'image': ruta} for ruta in rutas
        ] + [{'type': 'text', 'text': texto_user}]
    }
]
text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
images = [Image.open(ruta).convert('RGB') for ruta in rutas]
inputs = tokenizer(text=text_input, images=images, return_tensors='pt', truncation=True, max_length=4096).to('cuda')
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=False)
sample_output = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('--- Sample inference result ---')
print(sample_output)
